# 🏥 Stack-Based Ensemble TinyML vs SMEESI
## Dataset: IoMT-TrafficData — IP-Based Flows Pre-Processed (Multiclass)
## DOI: 10.1109/ACCESS.2024.3437214 | Zenodo: 10.5281/zenodo.8116337

### Label Strategy
| Column | Role |
|---|---|
| `traffic` | **Multiclass label** (9 classes) — used as target |
| `is_attack` | Dropped — it is a derived binary version of `traffic` |

### 9 Traffic Classes
| Class | Type |
|---|---|
| `normal` | Benign |
| `rudeadyet` | DoS attack |
| `slowloris` | DoS attack |
| `slowread` | DoS attack |
| `apachekiller` | DoS attack |
| `netscan` | Network scan |
| `arpspoofing` | ARP spoofing |
| `camoverflow` | Camera overflow |
| `mqttmalaria` | MQTT attack |

### Models
- **Stack Ensemble TinyML**: Autoencoder + VAE + XGBoost + TinyML Meta-Learner (softmax)
- **SMEESI**: Sustainable Micro-Neural Energy Efficient Security Intelligence (softmax)
- **Metrics**: Accuracy, Precision, Recall, F1 (weighted), ROC-AUC (OvR), Latency, TFLite INT8

## CELL 1 — Install Dependencies

In [ ]:
!pip install -q xgboost scikit-learn imbalanced-learn tensorflow numpy pandas matplotlib seaborn psutil tabulate
print('✅ All packages installed.')

## CELL 2 — Mount Drive & Import Libraries

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, time, warnings, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, regularizers
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score
)
from xgboost import XGBClassifier
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter
warnings.filterwarnings('ignore')
np.random.seed(42)
tf.random.set_seed(42)

# ---------------------------------------------------------------
# ⚙️  SET THIS PATH to your Google Drive folder.
#
#   MyDrive/IoMT_TrafficData/
#     IP-Based Flows Pre-Processed Train.csv   (173 MB)
#     IP-Based Flows Pre-Processed Test.csv    ( 57 MB)
# ---------------------------------------------------------------
BASE_PATH  = '/content/drive/MyDrive/IoMT_TrafficData'
OUTPUT_DIR = '/content/drive/MyDrive/IoMT_TrafficData/results_multiclass'
os.makedirs(OUTPUT_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(BASE_PATH, 'IP-Based Flows Pre-Processed Train.csv')
TEST_PATH  = os.path.join(BASE_PATH, 'IP-Based Flows Pre-Processed Test.csv')

# ---------------------------------------------------------------
# Column config
# traffic  = multiclass label (9 classes) — THIS IS OUR TARGET
# is_attack = binary derived column — DROPPED to avoid leakage
# service, history_originator, history_responder = categorical features
# ---------------------------------------------------------------
LABEL_COL  = 'traffic'     # 9-class target
DROP_COLS  = ['is_attack']  # derived binary — must drop to avoid leakage
CAT_COLS   = ['service', 'history_originator', 'history_responder']  # traffic is now label, not feature

print('✅ Libraries loaded. Drive mounted.')
print(f'Train : {TRAIN_PATH}')
print(f'Test  : {TEST_PATH}')
print(f'Output: {OUTPUT_DIR}')

## CELL 3 — Load Pre-Split Train & Test CSVs

In [ ]:
print('Loading TRAIN CSV...')
df_train = pd.read_csv(TRAIN_PATH, low_memory=False)
print(f'  Shape  : {df_train.shape[0]:,} rows x {df_train.shape[1]} columns')
print(f'  Missing: {df_train.isnull().sum().sum():,}')

print('\nLoading TEST CSV...')
df_test = pd.read_csv(TEST_PATH, low_memory=False)
print(f'  Shape  : {df_test.shape[0]:,} rows x {df_test.shape[1]} columns')
print(f'  Missing: {df_test.isnull().sum().sum():,}')

# Confirm traffic classes
print(f'\ntraffic column unique values (train): {sorted(df_train[LABEL_COL].dropna().unique())}')
print(f'traffic column unique values (test) : {sorted(df_test[LABEL_COL].dropna().unique())}')

print(f'\nAll columns: {list(df_train.columns)}')
print('\n✅ Both CSVs loaded.')

## CELL 4 — EDA: Class Distribution & Feature Overview

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('IoMT-TrafficData — IP Flows Multiclass EDA', fontsize=14, fontweight='bold')
palette = sns.color_palette('tab10', 9)

# Train class distribution
tr_dist = df_train[LABEL_COL].value_counts().sort_values(ascending=False)
bars = axes[0].bar(tr_dist.index, tr_dist.values, color=palette)
axes[0].set_title('Train — Traffic Class Distribution', fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)
for bar, v in zip(bars, tr_dist.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.01,
                 f'{v:,}', ha='center', fontsize=8, fontweight='bold')

# Test class distribution
te_dist = df_test[LABEL_COL].value_counts().sort_values(ascending=False)
bars2 = axes[1].bar(te_dist.index, te_dist.values,
                    color=palette[:len(te_dist)], alpha=0.85)
axes[1].set_title('Test — Traffic Class Distribution', fontweight='bold')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)
for bar, v in zip(bars2, te_dist.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.01,
                 f'{v:,}', ha='center', fontsize=8, fontweight='bold')

# Pie chart — train proportions
axes[2].pie(tr_dist.values, labels=tr_dist.index,
            autopct='%1.1f%%', colors=palette,
            startangle=140, textprops={'fontsize': 8})
axes[2].set_title('Train — Class Proportions', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'eda_class_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

print('Train class counts:')
print(tr_dist.to_string())
print(f'\nTest class counts:')
print(te_dist.to_string())

## CELL 5 — Feature Engineering & Preprocessing
> **Label**: `traffic` column — 9 classes, label-encoded (0–8)  
> **Dropped**: `is_attack` — derived binary of `traffic` — dropping prevents leakage  
> **Categorical features**: `service`, `history_originator`, `history_responder` — label-encoded  
> **Note**: `traffic` is now the label so it is NOT in `CAT_COLS`

In [ ]:
def preprocess(df_tr, df_te, label_col, drop_cols, cat_cols):
    # Extract labels
    y_train_raw = df_tr[label_col].values
    y_test_raw  = df_te[label_col].values

    # Drop label + leakage columns
    all_drop = [label_col] + [c for c in drop_cols if c in df_tr.columns]
    print(f'  Dropping from features: {all_drop}')
    X_tr = df_tr.drop(columns=all_drop).copy()
    X_te = df_te.drop(columns=all_drop).copy()

    # Label-encode categorical string features
    cat_encoders = {}
    for col in cat_cols:
        if col not in X_tr.columns:
            print(f'  WARNING: "{col}" not found — skipping')
            continue
        le_cat = LabelEncoder()
        X_tr[col] = le_cat.fit_transform(X_tr[col].astype(str))
        known = set(le_cat.classes_)
        X_te[col] = X_te[col].astype(str).apply(
            lambda x: int(le_cat.transform([x])[0]) if x in known else 0
        )
        cat_encoders[col] = le_cat
        print(f'  Encoded "{col}": {len(le_cat.classes_)} values → {list(le_cat.classes_)}')

    # Clean inf / NaN
    X_tr.replace([np.inf, -np.inf], np.nan, inplace=True)
    X_te.replace([np.inf, -np.inf], np.nan, inplace=True)
    X_tr.fillna(0, inplace=True)
    X_te.fillna(0, inplace=True)

    feat_names = list(X_tr.columns)
    X_tr = X_tr.values.astype(np.float32)
    X_te = X_te.values.astype(np.float32)

    # Label-encode target (traffic → 0–8)
    le_label = LabelEncoder()
    y_train  = le_label.fit_transform(y_train_raw)
    y_test   = le_label.transform(y_test_raw)

    # StandardScaler — fit on train only
    scaler = StandardScaler()
    X_tr   = scaler.fit_transform(X_tr)
    X_te   = scaler.transform(X_te)

    return X_tr, X_te, y_train, y_test, le_label, scaler, cat_encoders, feat_names

print('Preprocessing...')
X_tr_full, X_test_scaled, y_tr_full, y_test_all, \
    le_label, scaler, cat_encoders, feature_names = \
    preprocess(df_train, df_test, LABEL_COL, DROP_COLS, CAT_COLS)

n_features  = X_tr_full.shape[1]
NUM_CLASSES = len(le_label.classes_)
CLASS_NAMES = list(le_label.classes_)

print(f'\n✅ Preprocessing complete')
print(f'   Train shape    : {X_tr_full.shape}')
print(f'   Test shape     : {X_test_scaled.shape}')
print(f'   Features       : {n_features}')
print(f'   Classes ({NUM_CLASSES})    : {CLASS_NAMES}')
print(f'   Label mapping  : {dict(zip(CLASS_NAMES, le_label.transform(CLASS_NAMES)))}')

## CELL 6 — Undersample Training Set
> Balance all 9 classes to equal samples for fair multiclass training.  
> Cap at **5,500 per class** → ~49,500 total (close to 50k).  
> Classes with fewer than 5,500 samples keep ALL their samples.

In [ ]:
SAMPLES_PER_CLASS = 5_500   # 9 classes × 5,500 = ~49,500 total

print('Train class counts before undersampling:')
cc = Counter(y_tr_full)
for k, v in sorted(cc.items()):
    print(f'  {CLASS_NAMES[k]:15s} (id={k}): {v:,}')

sampling_strategy = {cls: min(cnt, SAMPLES_PER_CLASS) for cls, cnt in cc.items()}
rus    = RandomUnderSampler(sampling_strategy=sampling_strategy, random_state=42)
X_res, y_res = rus.fit_resample(X_tr_full, y_tr_full)

print(f'\n✅ After undersampling: {X_res.shape[0]:,} samples')
rc = Counter(y_res)
for k, v in sorted(rc.items()):
    print(f'  {CLASS_NAMES[k]:15s}: {v:,}')

# 80/20 train/val split
X_train, X_val, y_train, y_val = train_test_split(
    X_res, y_res, test_size=0.20, random_state=42, stratify=y_res
)
print(f'\n   Train : {X_train.shape}')
print(f'   Val   : {X_val.shape}')
print(f'   Test  : {X_test_scaled.shape}  (full author split — untouched)')

# Feature selector for SMEESI — top-30 for 9-class task
K_FEATURES = 30
selector   = SelectKBest(f_classif, k=K_FEATURES)
selector.fit(X_train, y_train)

scores  = selector.scores_
top_idx = np.argsort(scores)[::-1][:K_FEATURES]
print(f'\nTop-{K_FEATURES} features selected for SMEESI:')
for rank, idx in enumerate(top_idx, 1):
    print(f'  {rank:2d}. {feature_names[idx]:40s}  F={scores[idx]:.1f}')

del df_train, df_test, X_tr_full
gc.collect()

## CELL 7 — Visualize Class Balance After Undersampling

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Multiclass Balance After Undersampling', fontsize=13, fontweight='bold')
palette = sns.color_palette('tab10', NUM_CLASSES)

# After undersampling
rv = [rc[k] for k in sorted(rc)]
bars = axes[0].bar(CLASS_NAMES, rv, color=palette)
axes[0].set_title('Train — After Undersampling', fontweight='bold')
axes[0].set_ylabel('Samples')
axes[0].tick_params(axis='x', rotation=45)
for bar, v in zip(bars, rv):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.01,
                 f'{v:,}', ha='center', fontsize=9)

# Top features
top_names  = [feature_names[i] for i in top_idx]
top_scores = [scores[i] for i in top_idx]
yp = np.arange(K_FEATURES)
axes[1].barh(yp, top_scores[::-1], color=sns.color_palette('coolwarm_r', K_FEATURES))
axes[1].set_yticks(yp)
axes[1].set_yticklabels(top_names[::-1], fontsize=8)
axes[1].set_title(f'Top-{K_FEATURES} Features by F-Score (SMEESI)', fontweight='bold')
axes[1].set_xlabel('F-Score')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'class_balance_features.png'), dpi=150, bbox_inches='tight')
plt.show()

## CELL 8 — Stack Ensemble: Component 1 — Autoencoder (AE)

In [ ]:
def build_autoencoder(input_dim, latent_dim=32):
    inp = keras.Input(shape=(input_dim,))
    x = layers.Dense(128, activation='relu')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    encoded = layers.Dense(latent_dim, activation='relu', name='bottleneck')(x)
    x = layers.Dense(64,        activation='relu')(encoded)
    x = layers.Dense(128,       activation='relu')(x)
    decoded = layers.Dense(input_dim, activation='linear')(x)
    autoencoder = Model(inp, decoded, name='Autoencoder')
    encoder     = Model(inp, encoded, name='AE_Encoder')
    autoencoder.compile(optimizer='adam', loss='mse')
    return autoencoder, encoder

print(f'Building Autoencoder (input_dim={n_features}, latent_dim=32)...')
ae_model, ae_encoder = build_autoencoder(n_features, latent_dim=32)
ae_model.summary()

cb_ae = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
t0 = time.time()
ae_hist = ae_model.fit(
    X_train, X_train,
    validation_data=(X_val, X_val),
    epochs=30, batch_size=256, callbacks=[cb_ae], verbose=1
)
ae_train_time = time.time() - t0

ae_train_enc  = ae_encoder.predict(X_train,       batch_size=512, verbose=0)
ae_val_enc    = ae_encoder.predict(X_val,         batch_size=512, verbose=0)
ae_test_enc   = ae_encoder.predict(X_test_scaled, batch_size=512, verbose=0)

ae_train_recon = ae_model.predict(X_train,       batch_size=512, verbose=0)
ae_val_recon   = ae_model.predict(X_val,         batch_size=512, verbose=0)
ae_test_recon  = ae_model.predict(X_test_scaled, batch_size=512, verbose=0)
ae_train_err   = np.mean((X_train       - ae_train_recon)**2, axis=1, keepdims=True)
ae_val_err     = np.mean((X_val         - ae_val_recon)**2,   axis=1, keepdims=True)
ae_test_err    = np.mean((X_test_scaled - ae_test_recon)**2,  axis=1, keepdims=True)

print(f'\n✅ AE trained in {ae_train_time:.1f}s | Latent: {ae_train_enc.shape[1]}D')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(ae_hist.history['loss'],     label='Train')
axes[0].plot(ae_hist.history['val_loss'], label='Val', linestyle='--')
axes[0].set_title('AE Reconstruction Loss'); axes[0].legend()

# Recon error per class
for cls_id in range(NUM_CLASSES):
    mask = y_train == cls_id
    if mask.sum() > 0:
        axes[1].hist(ae_train_err[mask].flatten(), bins=60, alpha=0.5,
                     label=CLASS_NAMES[cls_id])
axes[1].set_title('AE Reconstruction Error by Class')
axes[1].set_xlabel('MSE'); axes[1].legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'ae_analysis.png'), dpi=150)
plt.show()

## CELL 9 — Stack Ensemble: Component 2 — VAE

In [ ]:
def build_vae(input_dim, latent_dim=16):
    inp = keras.Input(shape=(input_dim,))
    h = layers.Dense(64, activation='relu')(inp)
    h = layers.BatchNormalization()(h)
    z_mean    = layers.Dense(latent_dim, name='z_mean')(h)
    z_log_var = layers.Dense(latent_dim, name='z_log_var')(h)
    def sampling(args):
        z_m, z_lv = args
        return z_m + tf.exp(0.5 * z_lv) * tf.random.normal(shape=tf.shape(z_m))
    z = layers.Lambda(sampling, name='z')([z_mean, z_log_var])
    encoder = Model(inp, [z_mean, z_log_var, z], name='VAE_Encoder')

    dec_inp = keras.Input(shape=(latent_dim,))
    d = layers.Dense(64,        activation='relu')(dec_inp)
    d = layers.Dense(input_dim, activation='linear')(d)
    decoder = Model(dec_inp, d, name='VAE_Decoder')

    class VAE(Model):
        def __init__(self, enc, dec):
            super().__init__()
            self.enc = enc; self.dec = dec
            self.total_loss_tracker = keras.metrics.Mean(name='total_loss')
            self.recon_loss_tracker = keras.metrics.Mean(name='recon_loss')
            self.kl_loss_tracker    = keras.metrics.Mean(name='kl_loss')
        @property
        def metrics(self):
            return [self.total_loss_tracker, self.recon_loss_tracker, self.kl_loss_tracker]
        def train_step(self, data):
            x = data[0] if isinstance(data, tuple) else data
            with tf.GradientTape() as tape:
                z_m, z_lv, z_s = self.enc(x, training=True)
                recon      = self.dec(z_s, training=True)
                recon_loss = tf.reduce_mean(tf.reduce_sum(tf.square(x - recon), axis=1))
                kl_loss    = -0.5 * tf.reduce_mean(
                    tf.reduce_sum(1 + z_lv - tf.square(z_m) - tf.exp(z_lv), axis=1))
                total_loss = recon_loss + 0.01 * kl_loss
            grads = tape.gradient(total_loss, self.trainable_variables)
            self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
            self.total_loss_tracker.update_state(total_loss)
            self.recon_loss_tracker.update_state(recon_loss)
            self.kl_loss_tracker.update_state(kl_loss)
            return {m.name: m.result() for m in self.metrics}

    vae = VAE(encoder, decoder)
    vae.compile(optimizer='adam')
    return vae, encoder, decoder

print(f'Building VAE (input_dim={n_features}, latent_dim=16)...')
vae_model, vae_encoder, vae_decoder = build_vae(n_features, latent_dim=16)

t0 = time.time()
vae_hist = vae_model.fit(X_train, epochs=20, batch_size=256, verbose=1)
vae_train_time = time.time() - t0

vae_train_z, _, _ = vae_encoder.predict(X_train,       batch_size=512, verbose=0)
vae_val_z,   _, _ = vae_encoder.predict(X_val,         batch_size=512, verbose=0)
vae_test_z,  _, _ = vae_encoder.predict(X_test_scaled, batch_size=512, verbose=0)

vae_train_recon = vae_decoder.predict(vae_train_z, batch_size=512, verbose=0)
vae_val_recon   = vae_decoder.predict(vae_val_z,   batch_size=512, verbose=0)
vae_test_recon  = vae_decoder.predict(vae_test_z,  batch_size=512, verbose=0)
vae_train_err   = np.mean((X_train       - vae_train_recon)**2, axis=1, keepdims=True)
vae_val_err     = np.mean((X_val         - vae_val_recon)**2,   axis=1, keepdims=True)
vae_test_err    = np.mean((X_test_scaled - vae_test_recon)**2,  axis=1, keepdims=True)

print(f'✅ VAE trained in {vae_train_time:.1f}s | Latent: 16D')

plt.figure(figsize=(8, 4))
plt.plot(vae_hist.history['total_loss'], label='Total Loss')
plt.plot(vae_hist.history['recon_loss'], label='Recon Loss')
plt.plot(vae_hist.history['kl_loss'],    label='KL Loss')
plt.title('VAE Training Loss'); plt.xlabel('Epoch')
plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'vae_loss.png'), dpi=150)
plt.show()

## CELL 10 — Stack Ensemble: Component 3 — XGBoost (Multiclass)

In [ ]:
# AE latent (32) + VAE latent (16) + AE recon err (1) + VAE recon err (1) = 50 features
X_train_xgb = np.hstack([ae_train_enc, vae_train_z, ae_train_err, vae_train_err])
X_val_xgb   = np.hstack([ae_val_enc,   vae_val_z,   ae_val_err,   vae_val_err])
X_test_xgb  = np.hstack([ae_test_enc,  vae_test_z,  ae_test_err,  vae_test_err])
print(f'XGBoost stacked input shape: {X_train_xgb.shape}')

xgb_clf = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softprob',    # multiclass
    num_class=NUM_CLASSES,
    eval_metric='mlogloss',        # multiclass log loss
    random_state=42,
    n_jobs=-1,
    tree_method='hist',
    verbosity=0
)

print('Training XGBoost (multiclass)...')
t0 = time.time()
xgb_clf.fit(X_train_xgb, y_train,
            eval_set=[(X_val_xgb, y_val)], verbose=50)
xgb_train_time = time.time() - t0

y_xgb_val  = xgb_clf.predict(X_val_xgb)
y_xgb_test = xgb_clf.predict(X_test_xgb)
print(f'\n✅ XGBoost trained in {xgb_train_time:.1f}s')
print(f'   Val  Acc: {accuracy_score(y_val,      y_xgb_val)*100:.2f}%')
print(f'   Test Acc: {accuracy_score(y_test_all, y_xgb_test)*100:.2f}%')

## CELL 11 — Stack Ensemble: TinyML Meta-Learner (Softmax)

In [ ]:
# Meta features: AE latent (32) + VAE latent (16) + XGB proba (9) = 57
xgb_train_proba = xgb_clf.predict_proba(X_train_xgb)  # shape: (N, 9)
xgb_val_proba   = xgb_clf.predict_proba(X_val_xgb)
xgb_test_proba  = xgb_clf.predict_proba(X_test_xgb)

X_meta_train = np.hstack([ae_train_enc, vae_train_z, xgb_train_proba])
X_meta_val   = np.hstack([ae_val_enc,   vae_val_z,   xgb_val_proba])
X_meta_test  = np.hstack([ae_test_enc,  vae_test_z,  xgb_test_proba])
print(f'Meta-learner input shape: {X_meta_train.shape}')

# One-hot encode for categorical_crossentropy
y_train_cat = keras.utils.to_categorical(y_train, NUM_CLASSES)
y_val_cat   = keras.utils.to_categorical(y_val,   NUM_CLASSES)

def build_meta_learner(input_dim, num_classes):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(32, activation='relu'),
        layers.Dense(num_classes, activation='softmax')   # 9-class output
    ], name='TinyML_MetaLearner')
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

meta_learner = build_meta_learner(X_meta_train.shape[1], NUM_CLASSES)
meta_learner.summary()

cb_meta = EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True)
t0 = time.time()
meta_hist = meta_learner.fit(
    X_meta_train, y_train_cat,
    validation_data=(X_meta_val, y_val_cat),
    epochs=50, batch_size=128,
    callbacks=[cb_meta], verbose=1
)
meta_train_time = time.time() - t0

t_inf = time.time()
y_ensemble_proba = meta_learner.predict(X_meta_test, batch_size=512)  # shape: (N, 9)
ensemble_latency = (time.time() - t_inf) / len(X_meta_test) * 1000
y_ensemble_pred  = np.argmax(y_ensemble_proba, axis=1)

ensemble_acc  = accuracy_score(y_test_all,  y_ensemble_pred)
ensemble_prec = precision_score(y_test_all, y_ensemble_pred, average='weighted')
ensemble_rec  = recall_score(y_test_all,    y_ensemble_pred, average='weighted')
ensemble_f1   = f1_score(y_test_all,        y_ensemble_pred, average='weighted')
ensemble_auc  = roc_auc_score(y_test_all,   y_ensemble_proba,
                               multi_class='ovr', average='weighted')

print(f'\n✅ Stack Ensemble — Test Results (9-class)')
print(f'   Accuracy : {ensemble_acc*100:.2f}%')
print(f'   Precision: {ensemble_prec*100:.2f}% (weighted)')
print(f'   Recall   : {ensemble_rec*100:.2f}% (weighted)')
print(f'   F1-Score : {ensemble_f1*100:.2f}% (weighted)')
print(f'   ROC-AUC  : {ensemble_auc:.4f} (OvR weighted)')
print(f'   Latency  : {ensemble_latency:.4f} ms/sample')

## CELL 12 — Stack Ensemble: Confusion Matrix & Classification Report

In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))
cm = confusion_matrix(y_test_all, y_ensemble_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            ax=ax, annot_kws={'size': 9})
ax.set_title('Confusion Matrix — Stack Ensemble TinyML (9-class)\nIoMT-TrafficData IP Flows',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'cm_ensemble.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\nClassification Report — Stack Ensemble:')
print(classification_report(y_test_all, y_ensemble_pred, target_names=CLASS_NAMES))

## CELL 13 — Stack Ensemble: TFLite INT8 Quantization

In [ ]:
def quantize_to_tflite_int8(model, rep_data, out_path):
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    def rep_gen():
        for i in range(min(500, len(rep_data))):
            yield [rep_data[i:i+1].astype(np.float32)]
    converter.representative_dataset = rep_gen
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type  = tf.int8
    converter.inference_output_type = tf.int8
    model_bytes = converter.convert()
    with open(out_path, 'wb') as f:
        f.write(model_bytes)
    size_kb = os.path.getsize(out_path) / 1024
    print(f'  Saved: {out_path}  ({size_kb:.1f} KB)')
    return size_kb

print('Quantizing Meta-Learner -> INT8 TFLite...')
i8_ens = quantize_to_tflite_int8(
    meta_learner, X_meta_train[:500],
    os.path.join(OUTPUT_DIR, 'meta_learner_int8.tflite')
)

ae_size_kb        = ae_model.count_params()    * 4 / 1024
vae_size_kb       = vae_encoder.count_params() * 4 / 1024
meta_size_kb      = meta_learner.count_params()* 4 / 1024
ensemble_total_kb = ae_size_kb + vae_size_kb + meta_size_kb

print(f'\nEnsemble model sizes:')
print(f'  AE           : {ae_size_kb:.1f} KB  ({ae_model.count_params():,} params)')
print(f'  VAE encoder  : {vae_size_kb:.1f} KB  ({vae_encoder.count_params():,} params)')
print(f'  Meta-learner : {meta_size_kb:.1f} KB  ({meta_learner.count_params():,} params)')
print(f'  Total float32: {ensemble_total_kb:.1f} KB')
print(f'  INT8 TFLite  : {i8_ens:.1f} KB')

## CELL 14 — SMEESI Model (9-class Softmax)
> Top-30 features, L1+L2 regularization, 64→32→16→9 (softmax).  
> Designed for MCU-class IoMT edge deployment. Full INT8 TFLite.

In [ ]:
X_smeesi_train = selector.transform(X_train).astype(np.float32)
X_smeesi_val   = selector.transform(X_val).astype(np.float32)
X_smeesi_test  = selector.transform(X_test_scaled).astype(np.float32)
print(f'SMEESI input: {X_smeesi_train.shape[1]} features (top-{K_FEATURES})')

def build_smeesi(input_dim, num_classes):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation='relu',
                     kernel_regularizer=regularizers.l1_l2(l1=1e-4, l2=1e-4)),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(32, activation='relu',
                     kernel_regularizer=regularizers.l1_l2(l1=1e-4, l2=1e-4)),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        layers.Dense(16, activation='relu'),
        layers.Dense(num_classes, activation='softmax')   # 9-class output
    ], name='SMEESI')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

smeesi_model = build_smeesi(X_smeesi_train.shape[1], NUM_CLASSES)
smeesi_model.summary()

cb_sm = EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True)
t0 = time.time()
smeesi_hist = smeesi_model.fit(
    X_smeesi_train, y_train,
    validation_data=(X_smeesi_val, y_val),
    epochs=60, batch_size=128,
    callbacks=[cb_sm], verbose=1
)
smeesi_train_time = time.time() - t0

t_inf = time.time()
y_smeesi_proba = smeesi_model.predict(X_smeesi_test, batch_size=512)  # shape: (N, 9)
smeesi_latency = (time.time() - t_inf) / len(X_smeesi_test) * 1000
y_smeesi_pred  = np.argmax(y_smeesi_proba, axis=1)

smeesi_acc  = accuracy_score(y_test_all,  y_smeesi_pred)
smeesi_prec = precision_score(y_test_all, y_smeesi_pred, average='weighted')
smeesi_rec  = recall_score(y_test_all,    y_smeesi_pred, average='weighted')
smeesi_f1   = f1_score(y_test_all,        y_smeesi_pred, average='weighted')
smeesi_auc  = roc_auc_score(y_test_all,   y_smeesi_proba,
                              multi_class='ovr', average='weighted')
smeesi_size_kb = smeesi_model.count_params() * 4 / 1024

print(f'\n✅ SMEESI — Test Results (9-class)')
print(f'   Accuracy : {smeesi_acc*100:.2f}%')
print(f'   Precision: {smeesi_prec*100:.2f}% (weighted)')
print(f'   Recall   : {smeesi_rec*100:.2f}% (weighted)')
print(f'   F1-Score : {smeesi_f1*100:.2f}% (weighted)')
print(f'   ROC-AUC  : {smeesi_auc:.4f} (OvR weighted)')
print(f'   Latency  : {smeesi_latency:.4f} ms/sample')
print(f'   Model KB : {smeesi_size_kb:.1f} KB ({smeesi_model.count_params():,} params)')

## CELL 15 — SMEESI: Confusion Matrix & TFLite INT8

In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))
cm_s = confusion_matrix(y_test_all, y_smeesi_pred)
sns.heatmap(cm_s, annot=True, fmt='d', cmap='Oranges',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            ax=ax, annot_kws={'size': 9})
ax.set_title('Confusion Matrix — SMEESI (9-class)\nIoMT-TrafficData IP Flows',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'cm_smeesi.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\nClassification Report — SMEESI:')
print(classification_report(y_test_all, y_smeesi_pred, target_names=CLASS_NAMES))

print('\nQuantizing SMEESI -> INT8 TFLite...')
i8_sm = quantize_to_tflite_int8(
    smeesi_model, X_smeesi_train[:500],
    os.path.join(OUTPUT_DIR, 'smeesi_int8.tflite')
)

## CELL 16 — Training Curves Comparison

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Training Curves — IoMT IP Flows 9-class IDS', fontsize=14, fontweight='bold')

axes[0,0].plot(meta_hist.history['loss'],     label='Train', color='steelblue')
axes[0,0].plot(meta_hist.history['val_loss'], label='Val',   color='red', linestyle='--')
axes[0,0].set_title('Meta-Learner — Loss'); axes[0,0].legend()

axes[0,1].plot(meta_hist.history['accuracy'],     label='Train', color='steelblue')
axes[0,1].plot(meta_hist.history['val_accuracy'], label='Val',   color='red', linestyle='--')
axes[0,1].set_title('Meta-Learner — Accuracy'); axes[0,1].legend()

axes[1,0].plot(smeesi_hist.history['loss'],     label='Train', color='darkorange')
axes[1,0].plot(smeesi_hist.history['val_loss'], label='Val',   color='red', linestyle='--')
axes[1,0].set_title('SMEESI — Loss'); axes[1,0].legend()

axes[1,1].plot(smeesi_hist.history['accuracy'],     label='Train', color='darkorange')
axes[1,1].plot(smeesi_hist.history['val_accuracy'], label='Val',   color='red', linestyle='--')
axes[1,1].set_title('SMEESI — Accuracy'); axes[1,1].legend()

for ax in axes.flat:
    ax.set_xlabel('Epoch'); ax.set_ylabel('Value')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

## CELL 17 — Final Comparative Results Dashboard

In [ ]:
results = {
    'Model':               ['Stack Ensemble TinyML\n(AE+VAE+XGB+Meta)', 'SMEESI'],
    'Accuracy (%)':        [round(ensemble_acc*100,  2), round(smeesi_acc*100,  2)],
    'Precision (%)':       [round(ensemble_prec*100, 2), round(smeesi_prec*100, 2)],
    'Recall (%)':          [round(ensemble_rec*100,  2), round(smeesi_rec*100,  2)],
    'F1-Score (%)':        [round(ensemble_f1*100,   2), round(smeesi_f1*100,   2)],
    'ROC-AUC (OvR)':       [round(ensemble_auc,       4), round(smeesi_auc,       4)],
    'Latency (ms/sample)': [round(ensemble_latency,   4), round(smeesi_latency,   4)],
    'Model Size (KB)':     [round(ensemble_total_kb,  1), round(smeesi_size_kb,   1)],
    'TFLite INT8 (KB)':    [round(i8_ens, 1),              round(i8_sm, 1)],
    'Input Features':      [n_features,                    K_FEATURES],
    'Parameters':          [
        ae_model.count_params() + vae_encoder.count_params() + meta_learner.count_params(),
        smeesi_model.count_params()
    ],
    'Train Time (s)':      [
        round(ae_train_time + vae_train_time + xgb_train_time + meta_train_time, 1),
        round(smeesi_train_time, 1)
    ]
}

df_results = pd.DataFrame(results)
print('\n' + '='*85)
print('  FINAL RESULTS — IoMT-TrafficData IP Flows 9-Class IDS')
print('  Label: traffic column | Classes:', CLASS_NAMES)
print('  DOI: 10.1109/ACCESS.2024.3437214 | Zenodo: 10.5281/zenodo.8116337')
print('='*85)
print(df_results.to_string(index=False))
df_results.to_csv(os.path.join(OUTPUT_DIR, 'comparative_results.csv'), index=False)

# Bar chart
perf = ['Accuracy (%)', 'Precision (%)', 'Recall (%)', 'F1-Score (%)']
x = np.arange(len(perf)); w = 0.35
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Stack TinyML vs SMEESI — IoMT 9-Class IDS\nDOI: 10.1109/ACCESS.2024.3437214',
             fontsize=13, fontweight='bold')

b1 = axes[0].bar(x-w/2, [df_results.loc[0,m] for m in perf],
                 w, label='Stack Ensemble TinyML', color='steelblue', alpha=0.85)
b2 = axes[0].bar(x+w/2, [df_results.loc[1,m] for m in perf],
                 w, label='SMEESI', color='darkorange', alpha=0.85)
axes[0].set_xticks(x); axes[0].set_xticklabels(perf)
axes[0].set_ylim(0, 110); axes[0].set_ylabel('Score (%)')
axes[0].set_title('Performance Metrics (Weighted)'); axes[0].legend()
for bar in list(b1)+list(b2):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                 f'{bar.get_height():.1f}', ha='center', fontsize=8)

eff = ['Latency (ms/sample)', 'TFLite INT8 (KB)']
x2  = np.arange(len(eff))
e1 = axes[1].bar(x2-w/2, [df_results.loc[0,m] for m in eff],
                 w, label='Stack Ensemble TinyML', color='steelblue', alpha=0.85)
e2 = axes[1].bar(x2+w/2, [df_results.loc[1,m] for m in eff],
                 w, label='SMEESI', color='darkorange', alpha=0.85)
axes[1].set_xticks(x2); axes[1].set_xticklabels(eff)
axes[1].set_title('Efficiency Metrics (Lower = Better)'); axes[1].legend()
for bar in list(e1)+list(e2):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                 f'{bar.get_height():.3f}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'comparative_dashboard.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'\n✅ All results saved to: {OUTPUT_DIR}')

## CELL 18 — Deployment Summary

In [ ]:
print('\n' + '='*70)
print('   DEPLOYMENT ANALYSIS — IoMT IP Flows 9-Class IDS')
print('   DOI : 10.1109/ACCESS.2024.3437214')
print('   Data: Zenodo 10.5281/zenodo.8116337')
print('='*70)
print(f'''
┌─────────────────────────────────────────────────────────────────────┐
│              STACK-BASED ENSEMBLE TinyML                            │
│  AE (32-D latent) + VAE (16-D latent) + XGBoost + Meta-Learner    │
├─────────────────────────────────────────────────────────────────────┤
│  Task               : 9-class traffic classification               │
│  Output             : softmax over {NUM_CLASSES} classes                       │
│  Input features     : {n_features} (all encoded features)                  │
│  Deployment Target  : Edge servers, IoMT gateways, Raspberry Pi    │
│  Quantization       : TFLite INT8 (Meta-Learner component)         │
└─────────────────────────────────────────────────────────────────────┘
''')
print(f'''
┌──────────────────────────────────────────────────────────────┐
│                         SMEESI                               │
│  Sustainable Micro-Neural Energy Efficient Security Intel.   │
├──────────────────────────────────────────────────────────────┤
│  Task               : 9-class traffic classification         │
│  Output             : softmax over {NUM_CLASSES} classes                 │
│  Input features     : Top-{K_FEATURES} (SelectKBest f_classif)          │
│  Architecture       : 64 → 32 → 16 → {NUM_CLASSES} (softmax)            │
│  Regularization     : L1+L2 (1e-4) + Dropout 0.3/0.2        │
│  Deployment Target  : Microcontrollers, MCU-class IoMT nodes │
│  Quantization       : Full INT8 TFLite                       │
└──────────────────────────────────────────────────────────────┘
''')
print('\n--- Output Files ---')
for f in sorted(os.listdir(OUTPUT_DIR)):
    fp = os.path.join(OUTPUT_DIR, f)
    print(f'  {f:45s} {os.path.getsize(fp)/1024:>8.1f} KB')
print('\n' + '='*70)
print('✅ PROJECT COMPLETE — IoMT 9-Class Traffic IDS')
print('='*70)

## CELL 19 — Save All Models & Preprocessors

In [ ]:
import pickle

ae_model.save(os.path.join(OUTPUT_DIR,     'ae_model.h5'))
vae_encoder.save(os.path.join(OUTPUT_DIR,  'vae_encoder.h5'))
meta_learner.save(os.path.join(OUTPUT_DIR, 'meta_learner.h5'))
smeesi_model.save(os.path.join(OUTPUT_DIR, 'smeesi_model.h5'))
xgb_clf.save_model(os.path.join(OUTPUT_DIR,'xgb_model.json'))

with open(os.path.join(OUTPUT_DIR, 'scaler.pkl'),           'wb') as f: pickle.dump(scaler, f)
with open(os.path.join(OUTPUT_DIR, 'cat_encoders.pkl'),     'wb') as f: pickle.dump(cat_encoders, f)
with open(os.path.join(OUTPUT_DIR, 'label_encoder.pkl'),    'wb') as f: pickle.dump(le_label, f)
with open(os.path.join(OUTPUT_DIR, 'feature_selector.pkl'), 'wb') as f: pickle.dump(selector, f)
with open(os.path.join(OUTPUT_DIR, 'feature_names.pkl'),    'wb') as f: pickle.dump(feature_names, f)

print('✅ All models and preprocessors saved.')
print(f'   Location: {OUTPUT_DIR}')
print(f'   Class mapping: {dict(zip(CLASS_NAMES, le_label.transform(CLASS_NAMES)))}')